In [ ]:
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
from rotation.solver import initialize, solve_mixing
from rotation.symbolic import MASS, MIX, ANGLE

In [ ]:
# Mass info from online PDG (in GeV)
# https://pdglive.lbl.gov/Particle.action?node=S009
# https://pdglive.lbl.gov/Particle.action?node=S014
PDG = {
    MASS["pi"]: 0.1349768,
    MASS["eta"]: 0.547862,
}

In [ ]:
argument_order, system_of_equations, partials, _ = initialize()

known = {
    MIX["pi"]: PDG[MASS["pi"]],
    MASS["phi"]: 1.0,
    MIX["eta"]: PDG[MASS["eta"]],
    MIX["1x1"]: 0.1,
    MIX["1x2"]: 0.3,
}

unknown = {
    MIX["phi"]: 1.0,
    MASS["pi"]: PDG[MASS["pi"]],
    MASS["eta"]: PDG[MASS["eta"]],
    ANGLE["alpha"]: 0.0,
    ANGLE["beta"]: 0.0,
    ANGLE["gamma"]: 0.0,
}

In [ ]:
x_label = MASS["phi"]

solutions = []
for x in np.linspace(1e-2, 1e0,  num=201):
    known[x_label] = x
    try:
        solved = solve_mixing(
            system_of_equations,
            argument_order,
            unknown,
            known,
            partials,
        )
        solutions.append([known.copy(), solved.copy()])
    except ValueError as e:
        pass

In [ ]:
xs = np.array([sol[0][x_label] for sol in solutions])
# Reorganize solutions to be arrays of each unknown
data = {}
for sol in solutions:
    for y_label, data_point in sol[1].items():
        try:
            data[y_label].append(data_point)
        except KeyError:
            data[y_label] = [data_point]
for y_label, ys in data.items():
    data[y_label] = np.array(ys)
    data[y_label] = np.ma.array(ys, mask=data[y_label] < 0.0)

if not data:
    print("No data!")
else:
    # plotting
    fig, axs = plt.subplots(1,2, figsize=(10,6))
    
    for y_label, ys in data.items():
        if y_label in ANGLE.values():
            ax = axs[1]
        else:
            ax = axs[0]
        ax.plot(xs, ys.data, label=str(y_label), linewidth=0.5, marker=".")
    axs[0].set_ylabel("Mass [GeV]")
    axs[1].set_ylabel("Angle")
    title = []
    for k, v in solutions[0][0].items():
        if k is not x_label:
            title.append(f"{k}={v:.2e}")
    fig.suptitle(", ".join(title))
    for ax in axs.flatten():
        ax.set_xlabel(x_label)
        ax.legend()
        ax.grid(True)
    fig.show()